In [1]:
import os
import json
import pandas as pd
from tqdm import tqdm  # Uses standard fallback progress bar to avoid ipywidgets issues

# Folder configuration anchors
PAPERS_REPO_DIR = "/Users/salehalizade/Desktop/AXIOM/paperlists"
OUTPUT_JSONL_PATH = "axiom_nlp_comprehensive_dataset.jsonl"

# Domain Focus: Pure NLP/Computational Linguistics tracks across all historical archives
NLP_CONFERENCES = ["acl", "emnlp", "naacl", "coling"]

# Administrative and structural artifact filters to prevent data corruption
ADMIN_BLACKLIST = [
    "preface", "table of contents", "acknowledgment", "organisation", 
    "organization", "committee", "session", "index", "introduction by",
    "volume", "proceedings of", "reviewer acknowledgments"
]

In [2]:
def is_valid_nlp_paper(title):
    """Filters out empty items and administrative noise records."""
    if not title:
        return False
    
    title_lower = title.lower().strip()
    
    # Drop structural metadata rows
    if any(blacklisted_term in title_lower for blacklisted_term in ADMIN_BLACKLIST):
        return False
        
    # Exclude structural session headers or short non-paper blocks
    if len(title_lower.split()) < 3:
        return False
        
    return True

records_processed = 0

print(f"Beginning crawl from: {PAPERS_REPO_DIR}")

# Open output context in write-stream mode
with open(OUTPUT_JSONL_PATH, "w", encoding="utf-8") as jsonl_out:
    
    # Recursively descend into directory structure
    for root, dirs, files in os.walk(PAPERS_REPO_DIR):
        folder_name = os.path.basename(root).lower()
        
        # Enforce pure NLP track domain alignment
        if folder_name not in NLP_CONFERENCES:
            continue
            
        for file in tqdm(files, desc=f"Ingesting {folder_name.upper()} Archives"):
            if file.endswith(".json"):
                file_path = os.path.join(root, file)
                
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        raw_data = json.load(f)
                        
                    # Standardize schema mappings (handles both list arrays and dictionary hashes)
                    papers_iterable = raw_data.values() if isinstance(raw_data, dict) else raw_data
                    
                    for paper in papers_iterable:
                        # Defensive copy to avoid mutating cache structures
                        if not isinstance(paper, dict):
                            continue
                            
                        title = paper.get("title", "")
                        
                        # Validate entry cleanliness
                        if not is_valid_nlp_paper(title):
                            continue
                            
                        # Fallback parsing calculation for publication year if missing inside payload properties
                        extracted_year = paper.get("year") or "".join(filter(str.isdigit, file))
                        
                        # Deep-copy everything dynamically and layer normalized metrics over it
                        comprehensive_record = {**paper}
                        
                        comprehensive_record["venue"] = folder_name.upper()
                        comprehensive_record["year"] = int(extracted_year) if extracted_year else paper.get("year")
                        
                        # Stream straight to disk line-by-line
                        jsonl_out.write(json.dumps(comprehensive_record, ensure_ascii=False) + "\n")
                        records_processed += 1
                        
                except Exception as e:
                    print(f"Skipping unreadable payload at path [{file}]: {e}")

print(f"\nCompleted! Generated {records_processed} total NLP records in: '{OUTPUT_JSONL_PATH}'")


Beginning crawl from: /Users/salehalizade/Desktop/AXIOM/paperlists

Completed! Generated 0 total NLP records in: 'axiom_nlp_comprehensive_dataset.jsonl'


In [3]:
# Read the shared dataset file straight into a dataframe matrix
df = pd.read_json(OUTPUT_JSONL_PATH, lines=True)

# 1. Milestone Integrity Auditing
print(f"Total Papers Compiled: {len(df)}")
if len(df) >= 10000:
    print("✅ Target Met! Sufficient density reached for semantic indexing.")
else:
    print("⚠️ Target Alert! Volume falls below 10k baseline.")

# 2. Key Target Key Fields Integrity Checklist
critical_fields = ['id', 'author', 'bibtex', 'pdf', 'aff', 'github']
print("\n--- Feature Presence Verification ---")
for field in critical_fields:
    if field in df.columns:
        filled_percentage = (df[field].dropna() != "").sum() / len(df) * 100
        print(f"Field '{field}': Found in dataset (Data density: {filled_percentage:.2f}%)")
    else:
        print(f"Field '{field}': ❌ Missing from generated records")

# 3. Final Review of all extracted parameters
print("\n--- Extracted Data Sample ---")
df.head(2)

Total Papers Compiled: 0
⚠️ Target Alert! Volume falls below 10k baseline.

--- Feature Presence Verification ---
Field 'id': ❌ Missing from generated records
Field 'author': ❌ Missing from generated records
Field 'bibtex': ❌ Missing from generated records
Field 'pdf': ❌ Missing from generated records
Field 'aff': ❌ Missing from generated records
Field 'github': ❌ Missing from generated records

--- Extracted Data Sample ---


""


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame


In [5]:
print(list(df.columns))

[]


In [6]:
import pandas as pd

# Define the file you generated in the previous step
INPUT_JSONL_PATH = "FULL_DATASET.jsonl"

print(f"Loading dataset from {INPUT_JSONL_PATH}...")

# lines=True tells pandas to read the file newline-by-newline
df = pd.read_json(INPUT_JSONL_PATH, lines=True)

print(f"✅ Data loaded successfully! Shape: {df.shape}")
df.head(2)

Loading dataset from FULL_DATASET.jsonl...
✅ Data loaded successfully! Shape: (23002, 59)


,id,title,track,status,award,abstract,author,authorids,bibtex,pdf,...,confidence_avg,excitement_avg,reproducibility_avg,correctness_avg,replies_avg,authors#_avg,corr_rating_confidence,corr_rating_correctness,orcid,linkedin
0,2020.coling-main.519,"100,000 Podcasts: A Spoken English Document Co...",main,Main,0.0,Podcasts are a large and growing repository of...,Ann Clifton; Sravana Reddy; Yongze Yu; Aasish ...,/a/ann-clifton/; /s/sravana-reddy/; /y/yongze-...,"@inproceedings{clifton-etal-2020-100000,\n ...",https://aclanthology.org/2020.coling-main.519.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020.coling-main.113,A BERT-based Dual Embedding Model for Chinese ...,main,Main,0.0,Chinese idioms are special fixed phrases usual...,Minghuan Tan; Jing Jiang,/m/minghuan-tan/; /j/jing-jiang/,"@inproceedings{tan-jiang-2020-bert,\n title...",https://aclanthology.org/2020.coling-main.113.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df.columns

Index(['id', 'title', 'track', 'status', 'award', 'abstract', 'author',
       'authorids', 'bibtex', 'pdf', 'site', 'pdf_size', 'gs_citation',
       'gs_cited_by_link', 'gs_version_total', 'aff', 'aff_domain', 'email',
       'github', 'project', 'author_num', 'aff_unique_index',
       'aff_unique_norm', 'aff_unique_dep', 'aff_unique_url',
       'aff_unique_abbr', 'aff_campus_unique_index', 'aff_campus_unique',
       'aff_country_unique_index', 'aff_country_unique', 'venue', 'year',
       'tldr', 'keywords', 'primary_area', 'supplementary_material', 'gender',
       'homepage', 'dblp', 'google_scholar', 'or_profile', 'position',
       'reviewers', 'rating', 'confidence', 'excitement', 'reproducibility',
       'correctness', 'rating_avg', 'confidence_avg', 'excitement_avg',
       'reproducibility_avg', 'correctness_avg', 'replies_avg', 'authors#_avg',
       'corr_rating_confidence', 'corr_rating_correctness', 'orcid',
       'linkedin'],
      dtype='str')

FOR qdrant we need 

In [8]:
import re

# 1. Select the necessary columns for Qdrant payload filters & graph connections
columns_to_keep = ['id', 'title', 'abstract', 'venue', 'year']
qdrant_df = df[columns_to_keep].copy()

In [9]:
import collections
import re

# Combine titles and abstracts to analyze the entire textual corpus surface
# 1. Fill any missing data with empty strings, cast explicitly to string, and join them element-by-element
combined_series = df['title'].fillna("").astype(str) + " " + df['abstract'].fillna("").astype(str)

# 2. Flatten the entire corpus column down into one giant text block for character scanning
all_text = " ".join(combined_series)
# Define standard characters we EXPECT and want to keep (alphanumeric, spaces, basic punctuation)
# Anything outside this regex is flagged as a "strange element"
standard_chars_regex = r'[A-Za-z0-9\s.,!?()\'"\-\[\]:;_/+=%&<>*]'

print("Profiling corpus text for non-standard elements...")
strange_chars_counter = collections.Counter()

for text_block in all_text:
    if not isinstance(text_block, str):
        continue
    # Find all characters that DO NOT match our standard set
    strange_elements = re.findall(r'[^A-Za-z0-9\s.,!?()\'"\-\[\]:;_/+=%&<>*]', text_block)

    strange_chars_counter.update(strange_elements)

# Convert profiles to a readable summary table
profile_df = pd.DataFrame(strange_chars_counter.most_common(), columns=['Symbol/Element', 'Total Occurrences'])
print(f"Profiling complete! Found {len(profile_df)} unique non-standard elements.")

Profiling corpus text for non-standard elements...
Profiling complete! Found 193 unique non-standard elements.


In [10]:
from collections import Counter

# Combine all title and abstract strings into one giant corpus block
all_corpus_text = " ".join(df['title'].fillna("").astype(str) + " " + df['abstract'].fillna("").astype(str))

# Count characters that are NOT standard lowercase/uppercase english or baseline digits or space
anomalies = [char for char in all_corpus_text if not (char.isalnum() or char.isspace())]
anomaly_counts = Counter(anomalies)

# Convert into a clean dataframe sorted by the most frequent anomalies
profile_df = pd.DataFrame(anomaly_counts.most_common(), columns=['Character/Symbol', 'Total Found'])
print(f"Scan complete. Found {len(profile_df)} distinct unique non-alphanumeric characters.")

Scan complete. Found 92 distinct unique non-alphanumeric characters.


In [11]:
pd.set_option('display.max_rows', 100)
display(profile_df)

,Character/Symbol,Total Found
0,",",199447
1,.,183348
2,-,176798
3,),44468
4,(,42702
5,:,21148
6,/,13764
7,’,8580
8,%,6998
9,*,5910


In [12]:
glued_commas = df['abstract'].str.contains(r'[a-zA-Z],[a-zA-Z]').fillna(False).sum()
glued_periods = df['abstract'].str.contains(r'[a-zA-Z]\.[A-Z]').fillna(False).sum()
glued_semis = df['abstract'].str.contains(r'[a-zA-Z];[a-zA-Z]').fillna(False).sum()

print(f"Compressed Punctuation Profiles:\n- Glued commas (word,word): {glued_commas}\n- Glued sentence breaks (word.Word): {glued_periods}\n- Glued semicolons (word;word): {glued_semis}")

Compressed Punctuation Profiles:
- Glued commas (word,word): 128
- Glued sentence breaks (word.Word): 1481
- Glued semicolons (word;word): 9


In [13]:
# Isolate records that contain heavy combinations of your flagged characters
messy_sample = df[df['abstract'].str.contains(r'\\|\{', na=False)].head(2)

print("--- Inspecting Real Context of Messy Elements --- \n")
for idx, row in messy_sample.iterrows():
    print(f"VENUE: {row['venue']} | YEAR: {row['year']}")
    print(f"TITLE: {row['title']}")
    print(f"RAW TEXT PREVIEW:\n{row['abstract'][:500]}...\n")
    print("=" * 70 + "\n")

--- Inspecting Real Context of Messy Elements --- 

VENUE: EMNLP | YEAR: 2023
TITLE: Towards Informative Few-Shot Prompt with Maximum Information Gain for In-Context Learning
RAW TEXT PREVIEW:
Large Language models (LLMs) possess the capability to engage In-context Learning (ICL) by leveraging a few demonstrations pertaining to a new downstream task as conditions. However, this particular learning paradigm suffers from high instability stemming from substantial variances induced by factors such as the input distribution of selected examples, their ordering, and prompt formats. In this work, we demonstrate that even when all these factors are held constant, the random selection of exam...


VENUE: EMNLP | YEAR: 2023
TITLE: SpeechGPT: Empowering Large Language Models with Intrinsic Cross-Modal Conversational Abilities
RAW TEXT PREVIEW:
Multi-modal large language models are regarded as a crucial step towards Artificial General Intelligence~(AGI) and have garnered significant interest with

In [14]:
from pathlib import Path

OUTPUT_JSONL_PATH = "cleaned_dataset.jsonl"

def normalize_text(value):
    if value is None:
        return ""
    text = str(value)
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[{}]|\\[a-zA-Z]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_record(record):
    title = normalize_text(record.get("title"))
    abstract = normalize_text(record.get("abstract"))
    if not title or not abstract:
        return None
    title_lower = title.lower()
    if any(term in title_lower for term in ADMIN_BLACKLIST):
        return None
    if len(title.split()) < 3:
        return None
    cleaned = {**record}
    cleaned["title"] = title
    cleaned["abstract"] = abstract
    cleaned["venue"] = normalize_text(record.get("venue", "")).upper()
    if cleaned.get("year") is not None:
        try:
            cleaned["year"] = int(cleaned["year"])
        except Exception:
            pass
    return cleaned

rows = []
seen_ids = set()
dropped = {"duplicate_id": 0, "missing_id": 0, "invalid": 0}

with open("FULL_DATASET.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        paper_id = str(record.get("id", "")).strip()
        if not paper_id:
            dropped["missing_id"] += 1
            continue
        if paper_id in seen_ids:
            dropped["duplicate_id"] += 1
            continue
        cleaned = clean_record(record)
        if cleaned is None:
            dropped["invalid"] += 1
            continue
        seen_ids.add(paper_id)
        rows.append(cleaned)

cleaned_df = pd.DataFrame(rows)
with open(OUTPUT_JSONL_PATH, "w", encoding="utf-8") as out:
    for record in rows:
        out.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Cleaned rows: {len(cleaned_df)}")
print(f"Dropped duplicate ids: {dropped['duplicate_id']}")
print(f"Dropped missing ids: {dropped['missing_id']}")
print(f"Dropped invalid rows: {dropped['invalid']}")


Cleaned rows: 23002
Dropped duplicate ids: 0
Dropped missing ids: 0
Dropped invalid rows: 0


In [15]:
# Re-run the same QA checks on the cleaned dataset
df = cleaned_df.copy()

print(f"Total Papers Compiled After Cleaning: {len(df)}")
print("\n--- Key Field Check ---")
for field in ['id', 'author', 'bibtex', 'pdf', 'aff', 'github']:
    if field in df.columns:
        filled_percentage = (df[field].fillna('').astype(str).str.strip() != '').sum() / len(df) * 100
        print(f"Field '{field}': Data density {filled_percentage:.2f}%")
    else:
        print(f"Field '{field}': ❌ Missing")

print("\n--- Cleaning Impact Scan ---")
print("Empty abstracts:", (df['abstract'].fillna('').astype(str).str.strip() == '').sum())
print("Abstracts with latexish artifacts:", df['abstract'].fillna('').astype(str).str.contains(r'[{}\\$^_]', regex=True).sum())
print("Abstracts with extra whitespace:", df['abstract'].fillna('').astype(str).str.contains(r'\s{2,}', regex=True).sum())
print("Abstracts with urls:", df['abstract'].fillna('').astype(str).str.contains(r'http', case=False, regex=False).sum())

df.head(2)


Total Papers Compiled After Cleaning: 23002

--- Key Field Check ---
Field 'id': Data density 100.00%
Field 'author': Data density 100.00%
Field 'bibtex': Data density 99.99%
Field 'pdf': Data density 91.25%
Field 'aff': Data density 86.48%
Field 'github': Data density 30.78%

--- Cleaning Impact Scan ---
Empty abstracts: 0
Abstracts with latexish artifacts: 589
Abstracts with extra whitespace: 0
Abstracts with urls: 2912


,id,title,track,status,award,abstract,author,authorids,bibtex,pdf,...,confidence_avg,excitement_avg,reproducibility_avg,correctness_avg,replies_avg,authors#_avg,corr_rating_confidence,corr_rating_correctness,orcid,linkedin
0,2020.coling-main.519,"100,000 Podcasts: A Spoken English Document Co...",main,Main,False,Podcasts are a large and growing repository of...,Ann Clifton; Sravana Reddy; Yongze Yu; Aasish ...,/a/ann-clifton/; /s/sravana-reddy/; /y/yongze-...,"@inproceedings{clifton-etal-2020-100000,\n ...",https://aclanthology.org/2020.coling-main.519.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020.coling-main.113,A BERT-based Dual Embedding Model for Chinese ...,main,Main,False,Chinese idioms are special fixed phrases usual...,Minghuan Tan; Jing Jiang,/m/minghuan-tan/; /j/jing-jiang/,"@inproceedings{tan-jiang-2020-bert,\n title...",https://aclanthology.org/2020.coling-main.113.pdf,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# Profile the cleaned corpus for remaining unusual characters
combined_series = df['title'].fillna('').astype(str) + ' ' + df['abstract'].fillna('').astype(str)
all_text = ' '.join(combined_series)

strange_chars_counter = collections.Counter()
for text_block in all_text:
    if not isinstance(text_block, str):
        continue
    strange_elements = re.findall(r'[^A-Za-z0-9\s.,!?()\'"\-\[\]:;_/+=%&<>*]', text_block)
    strange_chars_counter.update(strange_elements)

profile_df = pd.DataFrame(strange_chars_counter.most_common(), columns=['Symbol/Element', 'Total Occurrences'])
print(f"Profiling complete on cleaned dataset. Found {len(profile_df)} unique non-standard elements.")
display(profile_df.head(25))


Profiling complete on cleaned dataset. Found 191 unique non-standard elements.


,Symbol/Element,Total Occurrences
0,’,8580
1,”,2354
2,“,2164
3,—,1567
4,–,1120
5,$,867
6,‘,506
7,@,247
8,\,242
9,×,210
